In [1]:
import os


import mdtraj as md
import nglview as nv

/Users/luismantilla/miniforge3/envs/agentem/lib/python3.11/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
t = md.load_dcd("../sims/traj.dcd", top="../sims/final.pdb")
t.image_molecules(inplace=True)  # wrap across PBC
t.center_coordinates()
t.save_dcd("../sims/traj_imaged.dcd")

# # then view sims/traj_imaged.dcd instead of sims/traj.dcd
v = nv.show_mdtraj(t)
v.clear_representations()
v.add_representation("cartoon", selection="protein")
# v.add_representation("ball+stick", selection="not protein and not water")  # ligand
v.center()

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


In [3]:
import mdtraj as md

t = md.load_pdb("../sims/final.pdb")

protein_atoms = [a.index for a in t.topology.atoms if a.residue.is_protein]
water_atoms = [a.index for a in t.topology.atoms if a.residue.is_water]
ion_resnames = {"NA", "K", "CL", "MG", "CA", "ZN"}  # extend if needed
ion_atoms = [a.index for a in t.topology.atoms if a.residue.name in ion_resnames]

# Your ligand (UNK with 9 atoms)
lig_atoms = []
for res in t.topology.residues:
    if res.name == "UNK" and len(list(res.atoms)) == 9:
        lig_atoms = [a.index for a in res.atoms]
        break

count = lambda idxs: len(idxs)
print("Total atoms:      ", t.topology.n_atoms)
print("Protein atoms:    ", count(protein_atoms))
print(
    "Water atoms:      ",
    count(water_atoms),
    f"(~{count(water_atoms) // 3} waters if TIP3P)",
)
print("Ion atoms:        ", count(ion_atoms))
print("Ligand atoms:     ", count(lig_atoms), "(DMS)")

# sanity: everything else
others = (
    set(range(t.topology.n_atoms))
    - set(protein_atoms)
    - set(water_atoms)
    - set(ion_atoms)
    - set(lig_atoms)
)
print("Other atoms:      ", len(others))

# Optional: quick residue histogram
from collections import Counter

print("Residue name counts:", Counter(res.name for res in t.topology.residues))

Total atoms:       75095
Protein atoms:     7548
Water atoms:       67422 (~22474 waters if TIP3P)
Ion atoms:         125
Ligand atoms:      9 (DMS)
Other atoms:       0
Residue name counts: Counter({'HOH': 22474, 'CL': 64, 'NA': 61, 'LEU': 53, 'PRO': 35, 'PHE': 32, 'GLU': 31, 'LYS': 30, 'ARG': 29, 'ILE': 28, 'GLY': 26, 'VAL': 25, 'ASP': 25, 'THR': 23, 'ASN': 19, 'SER': 19, 'ALA': 19, 'TYR': 18, 'HIS': 17, 'GLN': 13, 'MET': 9, 'CYS': 8, 'TRP': 2, 'UNK': 1})


In [6]:
import numpy as np
from scipy.spatial.distance import cdist

# ligand and protein atom coords (Å)
lig_xyz = t.xyz[0, lig_atoms, :] * 10.0
prot_xyz = t.xyz[0, protein_atoms, :] * 10.0

# compute nearest distances (Å) for each ligand atom to protein
dists = cdist(lig_xyz, prot_xyz).min(1)

print("Nearest distances (Å):", dists)
print("Mean pocket radius (Å):", dists.mean())
print("Max pocket radius (Å):", dists.max())

Nearest distances (Å): [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Mean pocket radius (Å): 0.0
Max pocket radius (Å): 0.0


In [8]:
from scipy.spatial.distance import cdist
import mdtraj as md

t = md.load_pdb("../sims/final.pdb")

# protein vs ligand atoms
protein_atoms = [a.index for a in t.topology.atoms if a.residue.is_protein]
lig_atoms = [a.index for a in t.topology.atoms if a.residue.name == "UNK"]

frame = 0
prot_xyz = t.xyz[frame, protein_atoms, :] * 10.0  # Å
lig_xyz = t.xyz[frame, lig_atoms, :] * 10.0  # Å

# distance of each protein atom to nearest ligand atom
dists = cdist(prot_xyz, lig_xyz).min(1)

# choose a cutoff for "pocket" (e.g. 5 Å)
cutoff = 5.0
pocket_mask = dists < cutoff
pocket_atoms = np.array(protein_atoms)[pocket_mask]

print(f"Protein atoms in {cutoff} Å pocket:", len(pocket_atoms))
print("First few atom IDs:", pocket_atoms[:10])

Protein atoms in 5.0 Å pocket: 109
First few atom IDs: [1378 1380 1381 1674 1676 1677 1678 1679 1680 1681]


In [13]:
t = md.load_dcd(
    "../sims/traj_imaged.dcd", top="../sims/final.pdb"
)  # or top="cyp2e1_fixed.pdb"
v = nv.show_mdtraj(t)
v.clear_representations()
v.add_representation("cartoon", selection="protein")
v.add_representation("ball+stick", selection="not protein and not water")  # ligand
v.center()
# v.stage.set_parameters(mouse_preset='coot')    # nicer mouse controls
# v.parameters = {"clipDist": 0}                 # avoid clipping
display(v)

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


NGLWidget(max_frame=24)

In [8]:
# Build a selection string for ONLY the ligand atoms
unk9 = [
    res
    for res in t.topology.residues
    if res.name == "UNK" and len(list(res.atoms)) == 9
][0]
lig_idx = [a.index for a in unk9.atoms]
lig_sel = "@" + ",".join(map(str, lig_idx))  # atom index selection

# Clear all old reps
v.clear_representations()

# Protein cartoon
v.add_representation("cartoon", selection="protein")

# Only this ligand
v.add_representation("ball+stick", selection=lig_sel)

# Optional water if you want (usually as transparent surface or lines)
v.add_representation("line", selection="water", opacity=0.2)

v.center()

In [9]:
import io
import numpy as np
import nglview as nv


def _box_pdb(center_A, size_A):
    cx, cy, cz = center_A
    sx, sy, sz = size_A
    hx, hy, hz = sx / 2, sy / 2, sz / 2
    C = [
        (cx - hx, cy - hy, cz - hz),
        (cx + hx, cy - hy, cz - hz),
        (cx + hx, cy + hy, cz - hz),
        (cx - hx, cy + hy, cz - hz),
        (cx - hx, cy - hy, cz + hz),
        (cx + hx, cy - hy, cz + hz),
        (cx + hx, cy + hy, cz + hz),
        (cx - hx, cy + hy, cz + hz),
    ]
    E = [
        (1, 2),
        (2, 3),
        (3, 4),
        (4, 1),
        (5, 6),
        (6, 7),
        (7, 8),
        (8, 5),
        (1, 5),
        (2, 6),
        (3, 7),
        (4, 8),
    ]
    s = io.StringIO()
    for i, (x, y, z) in enumerate(C, 1):
        s.write(
            f"HETATM{i:5d}  Q{i}  QM  A   1   {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n"
        )
    for a, b in E:
        s.write(f"CONECT{a:5d}{b:5d}\n")
    s.write("END\n")
    return s.getvalue()


def ligand_bbox_A(frame, padA=0.5):
    xyz = t.xyz[frame].copy()  # nm
    lig = xyz[lig_idx]  # nm
    com = lig.mean(0)
    L = t.unitcell_lengths[frame] if t.unitcell_lengths is not None else None
    if L is not None:
        lig = lig - np.round((lig - com) / L) * L  # PBC wrap near COM
    mins, maxs = lig.min(0), lig.max(0)
    centerA = ((mins + maxs) / 2.0) * 10.0  # Å
    sizeA = ((maxs - mins) * 10.0) + padA  # Å
    return centerA, sizeA


def add_qm_box(v, frame=0, padA=0.5, color="red"):
    centerA, sizeA = ligand_bbox_A(frame, padA)
    comp = v.add_component(nv.TextStructure(_box_pdb(centerA, sizeA)))
    comp.clear_representations()
    comp.add_representation("line", color=color)  # <- red, not grey
    return comp

In [10]:
from time import sleep

qm_comp = None
for frame in range(0, min(24, t.n_frames)):
    v.frame = frame
    if qm_comp is not None:
        v.remove_component(qm_comp)  # remove old box so it doesn't stack
    qm_comp = add_qm_box(v, frame=frame, padA=2.5, color="blue")
    sleep(0.1)

In [11]:
display(v)

NGLWidget(frame=23, max_frame=24)

In [12]:
from time import sleep

# to save time for this tutorial, we make a movie with only 50 frames
for frame in range(0, 25):
    # set frame to update coordinates
    v.frame = frame
    # make sure to let NGL spending enough time to update coordinates
    sleep(0.5)
    v.download_image(filename="../sims/frames/frame{}.png".format(frame))
    # make sure to let NGL spending enough time to render before going to next frame
    sleep(2.0)

In [32]:
# --- Box builder & overlay (Å units) ---
import numpy as np
import nglview as nv
import mdtraj as md

# ligand atom indices (not protein / not water)
# lig_idx = [a.index for a in t.topology.atoms if (not a.residue.is_protein) and (not a.residue.is_water)]
unk9 = [
    res
    for res in t.topology.residues
    if res.name == "UNK" and len(list(res.atoms)) == 9
][0]
lig_idx = [a.index for a in unk9.atoms]
lig_sel = "@" + ",".join(map(str, lig_idx))  # NGL atom-index selection


def _box_pdb_from_center_size(center_A, size_A):
    cx, cy, cz = center_A
    sx, sy, sz = size_A
    hx, hy, hz = sx / 2, sy / 2, sz / 2
    corners = [
        (cx - hx, cy - hy, cz - hz),
        (cx + hx, cy - hy, cz - hz),
        (cx + hx, cy + hy, cz - hz),
        (cx - hx, cy + hy, cz - hz),
        (cx - hx, cy - hy, cz + hz),
        (cx + hx, cy - hy, cz + hz),
        (cx + hx, cy + hy, cz + hz),
        (cx - hx, cy + hy, cz + hz),
    ]
    edges = [
        (1, 2),
        (2, 3),
        (3, 4),
        (4, 1),
        (5, 6),
        (6, 7),
        (7, 8),
        (8, 5),
        (1, 5),
        (2, 6),
        (3, 7),
        (4, 8),
    ]
    out = io.StringIO()
    for i, (x, y, z) in enumerate(corners, start=1):
        # valid PDB-ish line; don’t overthink columns
        out.write(
            f"HETATM{i:5d}  Q{i:1d}  QM  A   1   {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n"
        )
    for a, b in edges:
        out.write(f"CONECT{a:5d}{b:5d}\n")
    out.write("END\n")
    return out.getvalue()


def add_qm_box_component(v, frame=0, padding_A=1.0, color="red"):
    xyzA = t.xyz[frame, lig_idx, :] * 10.0
    mins, maxs = xyzA.min(0), xyzA.max(0)
    center = (mins + maxs) / 2.0
    size = (maxs - mins) + padding_A
    pdb_txt = _box_pdb_from_center_size(center, size)
    comp = v.add_component(nv.TextStructure(pdb_txt))
    comp.clear_representations()
    comp.add_representation("line", color=color)
    return comp

In [40]:
import numpy as np
import nglview as nv


def _box_pdb(center_A, size_A):
    cx, cy, cz = center_A
    sx, sy, sz = size_A
    hx, hy, hz = sx / 2, sy / 2, sz / 2
    C = [
        (cx - hx, cy - hy, cz - hz),
        (cx + hx, cy - hy, cz - hz),
        (cx + hx, cy + hy, cz - hz),
        (cx - hx, cy + hy, cz - hz),
        (cx - hx, cy - hy, cz + hz),
        (cx + hx, cy - hy, cz + hz),
        (cx + hx, cy + hy, cz + hz),
        (cx - hx, cy + hy, cz + hz),
    ]
    E = [
        (1, 2),
        (2, 3),
        (3, 4),
        (4, 1),
        (5, 6),
        (6, 7),
        (7, 8),
        (8, 5),
        (1, 5),
        (2, 6),
        (3, 7),
        (4, 8),
    ]
    s = io.StringIO()
    for i, (x, y, z) in enumerate(C, 1):
        s.write(
            f"HETATM{i:5d}  Q{i}  QM  A   1   {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n"
        )
    for a, b in E:
        s.write(f"CONECT{a:5d}{b:5d}\n")
    s.write("END\n")
    return s.getvalue()


def ligand_bbox_A(t, frame, lig_idx, padA=0.5):
    """Tight ligand box (Å). Makes ligand whole under PBC."""
    xyz = t.xyz[frame].copy()  # nm
    lig = xyz[lig_idx]  # nm
    com = lig.mean(0)
    # minimum-image wrap to keep atoms near COM
    L = t.unitcell_lengths[frame] if t.unitcell_lengths is not None else None
    if L is not None:
        lig = lig - np.round((lig - com) / L) * L
    mins, maxs = lig.min(0), lig.max(0)
    centerA = ((mins + maxs) / 2.0) * 10.0
    sizeA = ((maxs - mins) * 10.0) + padA  # add small padding (Å)
    return centerA, sizeA


def add_qm_box(v, frame=0, padA=0.5, color="red"):
    centerA, sizeA = ligand_bbox_A(t, frame, lig_idx, padA)
    comp = v.add_component(nv.TextStructure(_box_pdb(centerA, sizeA)))
    comp.clear_representations()
    comp.add_representation("line", color=color)
    return comp

In [35]:
from time import sleep

qm_comp = None
for frame in range(0, 24):
    # clear all shape/box components before loop

    v.frame = frame
    # refresh box
    if qm_comp is not None:
        v.remove_component(qm_comp)
    qm_comp = add_qm_box_component(v, frame=frame, padding_A=4, color="blue")

    # sleep(0.2)  # let it render
    # now grab your image (use your preferred capture method)
    # v.download_image(filename=f'../sims/frames/frame{frame}.png')  # if you use this path
    # sleep(0.5)

In [41]:
from time import sleep

qm = None
for frame in range(0, min(24, t.n_frames)):
    v.frame = frame
    if qm is not None:
        v.remove_component(qm)
    qm = add_qm_box(v, frame=frame, padA=0.5, color="red")  # smaller box
    sleep(0.1)

In [42]:
display(v)

NGLWidget(frame=23, max_frame=24, n_components=2)

In [43]:
from time import sleep

# to save time for this tutorial, we make a movie with only 50 frames
for frame in range(0, 25):
    # set frame to update coordinates
    v.frame = frame
    # make sure to let NGL spending enough time to update coordinates
    sleep(0.5)
    v.download_image(filename="../sims/frames/frame{}.png".format(frame))
    # make sure to let NGL spending enough time to render before going to next frame
    sleep(2.0)

KeyboardInterrupt: 

In [ ]:
# pip install pillow imageio-ffmpeg moviepy
import time
from PIL import Image
import mdtraj as md
import nglview as nv
from IPython.display import display

# --- load & show (frontend must be visible!) ---
t = md.load_dcd("../sims/traj_imaged.dcd", top="../sims/final.pdb")
v = nv.show_mdtraj(t)
v.clear_representations()
v.add_representation("cartoon", selection="protein")
v.add_representation("ball+stick", selection="not protein and not water")
v.center()
v._remote_call("setSize", target="Widget", args=["1280px", "720px"])
display(v)  # keep the tab focused while capturing
import base64


def grab_png(view, factor=1, antialias=True, timeout=5.0, poll=0.05, retries=2):
    """Return a PIL.Image from the current nglview frame."""
    PNG_SIG = b"\x89PNG\r\n\x1a\n"
    for _ in range(retries + 1):
        iw = view.render_image(
            factor=factor, antialias=antialias, trim=False, transparent=False
        )

        # wait for either iw.value (bytes) or view._image_data (base64 str)
        data_bytes = None
        t0 = time.time()
        while time.time() - t0 < timeout:
            val = getattr(iw, "value", None)  # bytes or None
            if isinstance(val, (bytes, bytearray)) and len(val) > 8:
                data_bytes = bytes(val)
                break
            b64 = getattr(view, "_image_data", None)  # 'data:image/png;base64,...'
            if isinstance(b64, str) and "base64," in b64:
                data_bytes = base64.b64decode(b64.split(",", 1)[1])
                break
            time.sleep(poll)

        iw.close()

        if data_bytes and data_bytes.startswith(PNG_SIG):
            return Image.open(io.BytesIO(data_bytes))

        # small pause before retry
        time.sleep(0.1)

    raise RuntimeError(
        "nglview did not provide a valid PNG. Make sure the widget is displayed and tab is active."
    )


# --- export some frames (subsample for speed) ---
os.makedirs("../sims/frames", exist_ok=True)
frames = []
for i in range(0, t.n_frames, 5):  # every 5th frame
    v.frame = i
    time.sleep(0.03)  # tiny settle helps
    img = grab_png(v, factor=1)  # use factor=2 for higher res (slower)
    path = f"../sims/frames/frame{i:05d}.png"
    img.save(path)
    frames.append(path)

NGLWidget(max_frame=24)

RuntimeError: nglview did not provide a valid PNG. Make sure the widget is displayed and tab is active.

In [ ]:
frames

[]

In [ ]:
import mdtraj as md
import nglview as nv
from nglview.contrib.movie import MovieMaker

t = md.load_dcd("../sims/traj.dcd", top="../sims/final.pdb")
v = nv.show_mdtraj(t)
v.clear_representations()
v.add_representation("cartoon", selection="protein")
v.add_representation("ball+stick", selection="not protein and not water")
v.center()
v.stage.set_parameters(mouse_preset="coot")
v.parameters = {"clipDist": 0}

# optional: set widget render size (affects movie resolution)
v._remote_call("setSize", target="Widget", args=["1280px", "720px"])

# make a .mov (H.264). set in_memory=True if you're on a remote notebook
mm = MovieMaker(
    v,
    output="traj.mov",  # ".gif" also works
    fps=5,
    start=0,
    stop=v.max_frame + 1,  # inclusive end -> add +1
    step=1,
    in_memory=True,  # good for small/medium videos; else keep False
    render_params={"factor": 2, "antialias": True, "trim": False, "transparent": False},
    moviepy_params={"codec": "libx264", "ffmpeg_params": ["-pix_fmt", "yuv420p"]},
)

prog = mm.make()  # shows a progress bar; file appears as 'traj.mov' when done
prog

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


IntProgress(value=0, description='Rendering ...', max=24)

In [ ]:
# pip install moviepy imageio imageio-ffmpeg
import mdtraj as md
import nglview as nv
from nglview.contrib.movie import MovieMaker
from IPython.display import display

# --- load + view ---
t = md.load_dcd("../sims/traj.dcd", top="../sims/final.pdb")
# (optional) subsample to speed up: t = t[::2]
v = nv.show_mdtraj(t)
v.clear_representations()
v.add_representation("cartoon", selection="protein")
v.add_representation("ball+stick", selection="not protein and not water")
v.center()
v._remote_call("setSize", target="Widget", args=["1280px", "720px"])  # 720p
display(v)  # MUST display so the frontend can render frames

print(f"Trajectory has {v.max_frame + 1} frames")

# --- make a quick movie (keeps it fast + reliable) ---
mm = MovieMaker(
    v,
    output="traj.mov",  # or "traj.gif"
    fps=24,
    start=0,
    stop=min(600, v.max_frame + 1),  # ~600 frames max for speed
    step=1,  # every other frame
)
mm.make()  # shows a progress bar; keeps running until "Done"

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


NGLWidget(max_frame=24)

Trajectory has 25 frames


IntProgress(value=0, description='Rendering ...', max=24)

In [ ]:
import matplotlib.image as mpimg
from time import sleep

imgs = []
for frame in range(0, 25):
    v.frame = frame
    sleep(0.2)  # let it update
    iw = v.render_image(factor=1, antialias=True, trim=False)
    sleep(0.2)  # let it finish rendering
    data = base64.b64decode(v._image_data)  # bytes
    img = mpimg.imread(io.BytesIO(data), format="PNG")
    mpimg.imsave(f"frame{frame:04d}.png", img)
    iw.close()  # free browser side resource

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x16c657100>

In [ ]:
from time import sleep

# to save time for this tutorial, we make a movie with only 50 frames
for frame in range(0, 50):
    # set frame to update coordinates
    view.frame = frame
    # make sure to let NGL spending enough time to update coordinates
    sleep(0.5)
    view.download_image(filename="0image{}.png".format(frame))
    # make sure to let NGL spending enough time to render before going to next frame
    sleep(2.0)

ModuleNotFoundError: No module named 'moviepy.editor'